In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
%run "./00_setup.ipynb"


In [ ]:
from domain.models import TwoStageConfig
from models.two_stage_return_model import PlaceTwoStageModel, WinTwoStageModel

cfg = TwoStageConfig()
win_model = WinTwoStageModel(cfg)
place_model = PlaceTwoStageModel(cfg)

print("## 2段階モデルの構造")
print("Stage A: P(hit) を予測 (binary classification)")
print("  - 全馬を学習データに使用")
print("  - label: finish_pos==1 (win) or finish_pos<=3 (place)")
print("")
print("Stage B: E(odds | hit) を予測 (L1 regression)")
print("  - 当該馬のみを学習データに使用")
print("  - target: win_odds_actual (win) or place_odds_actual (place)")
print("  - min 200 samples required")
print("")
print("EV = P × E")
print("  ev_win = p_win_pred × e_return_win_pred")


In [ ]:
print("""
## 2段階モデル学習手順

1. FeatureEngine.build_all() で特征量生成
2. MarketModel.train() + predict_and_calc_error() で log_error 計算
3. AbilityModel.train() + add_ability_probs() で能力値追加
4. WinTwoStageModel:
   a. train_hit_model(df) — P(win) 学習
   b. train_return_model(df) — E(odds|win) 学習 (winners only)
   c. predict_ev(df) — ev_win = P × E
5. EVCorrectionModel.train() + correct_ev() — P/E分解補正

実データでの学習には TrainingPipelineV5.run() を使用。
""")


In [ ]:
print("""
## 1段階モデル (ベースライン)

直接 EV を回帰する LightGBM:
  - features: 同じ特征量
  - target: win_odds_actual × (finish_pos == 1)
  - objective: regression (L1)

問題: ゼロ偏重
  - 多くの馬の actual_ev = 0 (非当选)
  - モデルは「全馬に低い予測値」を出力しがち
  → 高オーズ馬の的中時の払戻を見逃がす
""")


In [ ]:
print("""
## 比較項目

| 指標 | 2段階モデル | 1段階モデル |
|------|-----------|-----------|
| ゼロ偏重 | なし (PとEを分離) | あり (大部分がゼロ) |
| 高オーズ馬のEV | 適切に評価 | 過小評価 |
| AUC | P(hit) のみ評価可能 | EV全体で評価 |
| ROI | P×E で正確 | ゼロで過小評価 |

期待される結果:
  - 2段階モデルの方が ROI が高い (特に高オーズ带)
  - 1段階モデルは低オーズ带では良いが高オーズで壊れる
""")


In [ ]:
print("""
## 結論: 2段階 vs 1段階 ABテスト

2段階モデルの優位性:
  1. ゼロ偏重の排除 — PとEを独立に学習
  2. 高オーズ馬の適切な評価 — E(odds|hit) が大きな払戻を反映
  3. より正確な EV 推定 — 実運用での ROI 向上が期待される

実データでの検証には Notebook 11 (Hold-out) を使用。
""")
